In [138]:
import numpy as np
from torchvision.models import vgg11_bn,VGG11_BN_Weights 
import torch.nn as nn
from torch.utils.data import DataLoader
from scipy.special import lambertw
from sklearn.cluster import KMeans
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import torch
import numpy as np
import random
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10
from torchvision import transforms
from sklearn.metrics import confusion_matrix
import torch.nn.functional as F
import os
import pandas as pd


In [ ]:
# normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                                  std=[0.229, 0.224, 0.225])

# train_loader = torch.utils.data.DataLoader(
#     CIFAR10(root='./data', train=True, transform=transforms.Compose([
#         transforms.RandomHorizontalFlip(),
#         transforms.RandomCrop(32, 4),
#         transforms.ToTensor(),
#         normalize,
#     ]), download=True),
#     batch_size=16, shuffle=True,
#     num_workers=4, pin_memory=True)

In [96]:
class PretrainedVGG11(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.5):
        super(PretrainedVGG11, self).__init__()

        self.conv_block1 = self._make_conv_block(3, 64, 2)
        self.conv_block2 = self._make_conv_block(64, 128, 2)
        self.conv_block3 = self._make_conv_block(128, 256, 3)
        self.conv_block4 = self._make_conv_block(256, 512, 3)
        self.conv_block5 = self._make_conv_block(512, 512, 3)

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_conv_block(self, in_channels, out_channels, num_convs):
        layers = []
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1))
        layers.append(nn.BatchNorm2d(out_channels))  # BatchNorm after the convolution
        layers.append(nn.ReLU())
        for _ in range(num_convs - 1):
            layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1))
            layers.append(nn.BatchNorm2d(out_channels))  # BatchNorm after the convolution
            layers.append(nn.ReLU())
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        x = self.conv_block5(x)

        x = self.avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

In [126]:

class PretrainedVGG11(nn.Module):
    def __init__(self, num_classes):
        super(PretrainedVGG11, self).__init__()
        # Carica il modello VGG16 pre-addestrato
        self.vgg = vgg11_bn(weights=VGG11_BN_Weights.DEFAULT)
        for param in self.vgg.parameters():
            param.requires_grad = False
    
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        # Modifica l'ultimo strato della rete per il dataset CIFAR-10

        self.vgg.classifier[6] = nn.Linear(4096, num_classes)

    def forward(self, x):
        return self.vgg(x)

In [98]:


def calculate_confidence_score(global_model, dataloader, num_classes, lambda_reg, device,client_id,agent_wights,server_classes,round_federated):
    # print(f"calcolo confidenze agente {client_id}")
 
    global_model_ = PretrainedVGG11(num_classes=server_classes).to(device)
    
    global_model_.load_state_dict(agent_wights)
    criterion = nn.CrossEntropyLoss(reduction='none')
    global_model_.eval()
    global_model_.to(device)
    log_C = np.log(num_classes)
    #lambda regularization base = 0.5
    missing_class_agents = server_classes-num_classes
    if missing_class_agents >0:
        lambda_reg-= round(missing_class_agents/10,10)
        print(f"agente {client_id} reweight regularization missing {missing_class_agents} -> {lambda_reg}")        
    total_sigma = 0.0
    num_samples = 0
    CM = np.zeros((num_classes, num_classes), dtype=int)
    max_print = 0
    with torch.no_grad():
        for data in dataloader:
            # print(f"num_samples {num_samples}")
            images, labels = data
            images = images.to(device)
            labels = labels.to(device)
            # Predizioni del modello
            outputs = global_model_(images)
            # probabilities = torch.softmax(outputs, dim=1)
            # losses = -torch.log(probabilities[range(len(labels)), labels])
            # _, predicted = torch.max(outputs.data, 1)
            _, preds = torch.max(outputs, 1)           
            CM+=confusion_matrix(labels.cpu().numpy(), preds.cpu().numpy(),labels=range(num_classes))
            
            losses = criterion(outputs, labels)
            # print(losses)
            for loss_val in losses:
                loss = loss_val.item()
                # Calcola la confidenza
                
                max_term = max(-1 / np.exp(1), (loss - log_C) / lambda_reg)
               
                    # print(f"loss - logC / lambda reg -> {(loss - log_C) / lambda_reg}")
                    # print(f" -2 / np.exp(1) -> {-2 / np.exp(1)}")
                    # print(f"max term {max_term}")
                   
                W_input = 0.5 * max_term
                W_output = lambertw(W_input).real
                sigma_j = np.exp(-W_output)
                total_sigma += sigma_j
                num_samples += 1
                # if max_print <=5:
                #     max_print+=1
                #     print(f"w input {W_input}")
                #     print(f"w out {W_output}")
                #     print(f"sigma_j {sigma_j}")
            del images, labels, outputs
            torch.cuda.empty_cache()
    # print(f"total_sigma {total_sigma}")
    # print(f"num_samples {num_samples}")
    confidenza_media = total_sigma / num_samples if num_samples > 0 else 0
    # print(f"confidenza_media agente {client_id}: {confidenza_media}")
    pd.DataFrame(CM, index=[str(i) for i in range(num_classes)], columns=[str(i) for i in range(num_classes)]).to_csv(f"""./matrici_confusione/round_{round_federated}_agente_{client_id}.csv""")
    del global_model_
    return confidenza_media

In [99]:
torch.cuda.empty_cache()


In [100]:

def classify_clients_by_confidence(normalized_confidences):
    """
    Classifica i clienti come onesti o malevoli usando il clustering basato sui punteggi di confidenza normalizzati.

    Args:
    - normalized_confidences: Dizionario con ID cliente e confidenza normalizzata.

    Returns:
    - honest_clients: Set di ID clienti classificati come onesti.
    - malicious_clients: Set di ID clienti classificati come malevoli.
    """
    # Converti le confidenze normalizzate in una lista ordinata
    client_ids = list(normalized_confidences.keys())
    confidences = np.array(list(normalized_confidences.values())).reshape(-1, 1)

    # Applica k-means clustering con k=2
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    kmeans.fit(confidences)
    centroids = kmeans.cluster_centers_.flatten()

    # Ordina i centroidi (µlower e µupper)
    mu_lower, mu_upper = np.min(centroids), np.max(centroids)

    # Classifica i clienti
    honest_clients = set()
    malicious_clients = set()

    for client_id, sigma_i in normalized_confidences.items():
        # Classifica in base alla distanza dai centroidi
        if abs(sigma_i - mu_upper) < abs(sigma_i - mu_lower):
            honest_clients.add(client_id)
        else:
            malicious_clients.add(client_id)

    return honest_clients, malicious_clients


In [101]:


def federated_training_with_confidence(client_loaders, server_loader, num_agents=5, epochs=5, num_rounds=10, device='cuda',agent_classes=None,
                                       server_classes=10,lambda_reg = 0.25,resume_from_checkpoint = False,check_point_name = 'checkpoint.pth.tar'):
    """
    Addestramento federato con calcolo delle confidenze e identificazione dei client onesti.

    Args:
    - client_loaders: Lista di DataLoader per ciascun client.
    - server_loader: DataLoader per il server.
    - num_agents: Numero di client.
    - epochs: Numero di epoche per l'addestramento locale.
    - num_rounds: Numero di round di addestramento federato.
    - device: Dispositivo di esecuzione ('cpu' o 'cuda').

    Returns:
    - agents_confidence: Confidenze calcolate per ogni client.
    - aggregated_weights: Pesi aggregati del modello globale.
    """

    
    rounds_agent_confidences = {}

    # Modello globale
    global_model = PretrainedVGG11(num_classes=server_classes).to(device)
    global_weights = global_model.state_dict()
    already_resumed_from_checkpoint = False
    for round_num in range(num_rounds):
        
            
        print(f"\nRound {round_num + 1}/{num_rounds} iniziato...")

        agent_weights = {}
        data_lengths = {}
        clip_value = 5
        # Addestramento locale
        for client_id, loader in enumerate(client_loaders):
         
            print(f" Agente {client_id} inizia l'addestramento")
            # print(f"numero classi agente {client_id}: {agent_classes[client_id]}")
            local_model = PretrainedVGG11(num_classes=server_classes).to(device)
            # local_model.load_state_dict(global_weights)
            
            
            if resume_from_checkpoint and not already_resumed_from_checkpoint:
                checkpoint = load_checkpoint(check_point_name)
                local_model.load_state_dict(checkpoint['state_dict'])
                # optimizer.load_state_dict(checkpoint['optimizer'])
                print(f"Resuming from checkpoint agent {client_id}")
            else:
                local_model.load_state_dict(global_weights)
            optimizer = optim.SGD(local_model.parameters(), lr=0.01,weight_decay = 1e-3)
            criterion = nn.CrossEntropyLoss()
            for epoch in range(epochs):
                # print(f" Epoca {epoch} di {epochs}")

                local_model.train()
                for images, labels in loader:
                    images, labels = images.to(device), labels.to(device)
                    optimizer.zero_grad()
                    outputs = local_model(images)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(local_model.parameters(), clip_value)
                    optimizer.step()
                    del images, labels, outputs
                    torch.cuda.empty_cache()

                    
            # print(local_model.state_dict())
            # print(local_model.state_dict()['vgg.features.0.weight'][0])
            agent_weights[client_id]=local_model.state_dict()
            data_lengths[client_id] = len(loader.dataset)
        # Calcolo delle confidenze
        agent_confidences = {
            client_id: calculate_confidence_score(global_model, server_loader, agent_classes[client_id], lambda_reg, 
                                                  device,client_id,agent_weights[client_id],server_classes,round_num)
            for client_id in range(num_agents)
        }
        print(f"agents confidences {agent_confidences}")
     # Rescaling Min-Max delle confidenze
        conf_values = list(agent_confidences.values())
        min_conf = min(conf_values)
        max_conf = max(conf_values)
    
        if max_conf > min_conf:
            print("max > min conf")
            normalized_confidences = {
                client_id: (conf - min_conf) / (max_conf - min_conf)
                for client_id, conf in agent_confidences.items()
            }
        else:
            print("max <= min conf")
            normalized_confidences = {client_id: 1.0 for client_id in agent_confidences}

        print(f"Confidenze Normalizzate (Min-Max): {normalized_confidences}")
        rounds_agent_confidences[round_num + 1] = normalized_confidences

        # Classifica i client
        honest_clients, malicious_clients = classify_clients_by_confidence(normalized_confidences)
        print(f"Clienti onesti: {honest_clients}")
        print(f"Clienti malevoli: {malicious_clients}")
        # return agent_weights, data_lengths, normalized_confidences, honest_clients
        # Aggrega i pesi
        return agent_weights, data_lengths, normalized_confidences, honest_clients


        
        global_weights = reweighted_aggregation(agent_weights, data_lengths, normalized_confidences, honest_clients)
        global_model.load_state_dict(global_weights)
        already_resumed_from_checkpoint = True
        print(f"Checkpoint model round {round_num +1}")
        checkpoint = {
            'state_dict': global_model.state_dict()
            # 'optimizer': optimizer.state_dict(),
        }
        save_checkpoint(checkpoint,f"./checkpoint/{round_num + 1}_checkpoint.pth.tar")
        save_logs(honest_clients, malicious_clients,round_num+1,normalized_confidences,agent_confidences)
        print(f"Round {round_num + 1} completato.")

    return rounds_agent_confidences, global_weights


In [102]:
import os
import pandas as pd
import json

def save_logs(honest_clients, malicious_clients, round_num, normalized_confidences, agent_confidences, log_file='logs.csv'):
    log_file = f'./logs/{round_num}_logs.csv'
    # Verifica se il file di log esiste
    file_exists = os.path.isfile(log_file)
    
    # Crea un dizionario con i dati da salvare
    log_data = {
        'Round': round_num,
        'Honest_Clients': json.dumps(list(honest_clients)),
        'Malicious_Clients': json.dumps(list(malicious_clients)),
        'Normalized_Confidences': json.dumps(normalized_confidences),
        'Agent_Confidences': json.dumps(agent_confidences)
    }
    
    # Crea un DataFrame
    df = pd.DataFrame([log_data])
    
    # Scrivi o aggiungi al file CSV
    if not file_exists:
        # Scrivi con intestazioni
        df.to_csv(log_file, index=False)
        print(f"File di log creato e dati del round {round_num} salvati.")
    else:
        # Aggiungi senza intestazioni
        df.to_csv(log_file, mode='a', header=False, index=False)
        print(f"Dati del round {round_num} aggiunti al file di log esistente.")


In [103]:
# honest_clients =  {0, 1, 4, 6, 11, 12, 13, 20, 22, 23, 24}
# confidences ={0: 0.8287285246735777, 1: 0.5475098581042667,
#               2: 0.466357173974249, 3: 0.05533571887293809, 4: 0.6456192042044341, 
#               5: 0.2708017551605544, 6: 1.0, 7: 0.4202978473132509, 8: 0.3932329860718459, 
#               9: 0.10933497752543479, 10: 0.078671453001377, 11: 0.8287429467700059, 12: 0.6654338953545557, 
#               13: 0.6846731528976716, 14: 0.41164678583174374, 15: 0.4709850352342391, 16: 0.3134172338877447, 
#               17: 0.0, 18: 0.46320457474700516, 19: 0.44736670180039484, 20: 0.6639104077546569, 
#               21: 0.20338588150091036, 22: 0.5368151852795942, 23: 0.8715493317343778, 24: 0.6108713953497251}






In [104]:

def reweighted_aggregation(
    agent_weights: dict,
    data_lengths: dict,
    confidences: dict,
    honest_clients: list or set
):
    

    # 1. Calcolo dei pesi originali w_orig,i basati su data_lengths 
    sum_data_lengths = sum(data_lengths[cid] for cid in honest_clients)
    if sum_data_lengths == 0:
        raise ValueError("La somma delle lunghezze di tutti i client onesti è 0!")

    w_orig = {}
    for cid in honest_clients:
        w_orig[cid] = data_lengths[cid] / sum_data_lengths

    # 2. Normalizzazione delle confidenze per ottenere r_norm,i 
    sum_confidences = sum(confidences[cid] for cid in honest_clients)
    if sum_confidences == 0:
        
        print("Attenzione: la somma delle confidenze è 0.")
        r_norm = {cid: 1.0 / len(honest_clients) for cid in honest_clients}
    else:
        r_norm = {}
        for cid in honest_clients:
            r_norm[cid] = confidences[cid] / sum_confidences

    # --- 3. Calcolo dei pesi finali w_final,i ---
    w_final = {}
    for cid in honest_clients:
        w_final[cid] = r_norm[cid] * w_orig[cid]

    # --- 4. Aggregazione dei pesi ---
    
    numerator_sd = None
    denominator = 0.0

    for idx, cid in enumerate(honest_clients):
        local_sd = agent_weights[cid]  # state_dict del client i
        weight_factor = w_final[cid] * data_lengths[cid]
        denominator += weight_factor

        # Se è il primo client onesto, crea una copia profonda e scala
        if numerator_sd is None:
            numerator_sd = copy.deepcopy(local_sd)
            for key in numerator_sd:
                numerator_sd[key] = numerator_sd[key] * weight_factor
        else:
            # Somma pesata in tutti i layer
            for key in local_sd:
                numerator_sd[key] += local_sd[key] * weight_factor

    if denominator == 0:
        print("Attenzione: denominator = 0 durante l'aggregazione. Uso denominator=1 di default.")
        denominator = 1.0

    # Dividiamo per la somma dei pesi
    for key in numerator_sd:
        numerator_sd[key] /= denominator

     
    return numerator_sd


In [105]:


def make_federated_data(    dataset,    num_clients=5,    alpha=0.5,    server_fraction=0.1,    batch_size=32,    shuffle=True,    seed=42,
                       save_path = None):
 
    np.random.seed(seed)
    random.seed(seed)

    total_len = len(dataset)
    num_classes = 10  # se CIFAR-10
    os.makedirs(save_path, exist_ok=True)

    server_indices_path = os.path.join(save_path, 'server_indices.pt')
    client_indices_paths = [os.path.join(save_path, f'client_{i}_indices.pt') for i in range(num_clients)]
    
    indices_exist = os.path.exists(server_indices_path) and all(os.path.exists(path) for path in client_indices_paths)

    # ------------------------------------------------
    # 1) Selezioniamo i campioni per il server
    # ------------------------------------------------
    server_size = int(server_fraction * total_len)
    all_indices = list(range(total_len))
    
    if indices_exist:
        # Carica gli indici salvati
        server_indices = torch.load(server_indices_path)
        client_indices = []
        for path in client_indices_paths:
            client_indices.append(torch.load(path))
        print("[Info] Indici caricati dai file di split.")
    else:
        random.shuffle(all_indices)
        
        server_indices = all_indices[:server_size]
        client_indices_all = all_indices[server_size:]
        torch.save(server_indices, server_indices_path)
        print(f"[Info] Indici del server salvati in {server_indices_path}.")

        # ------------------------------------------------
        # 2) Raggruppiamo i rimanenti per classe
        # ------------------------------------------------
        indices_by_class = [[] for _ in range(num_classes)]
        for idx in client_indices_all:
            _, label = dataset[idx]
            indices_by_class[label].append(idx)

        # ------------------------------------------------
        # 3) Inizializziamo array per i client
        # ------------------------------------------------
        client_indices = [[] for _ in range(num_clients)]
    
        # Per tracciare la distribuzione {client: {classe: count}}
        agent_class_dist = {cid: {c: 0 for c in range(num_classes)} 
                            for cid in range(num_clients)}

        # ------------------------------------------------
        # 4) Partizione Dirichlet su ogni classe
        # ------------------------------------------------
        for c in range(num_classes):
            idxs_c = indices_by_class[c]
            random.shuffle(idxs_c)
            n_class = len(idxs_c)
            if n_class == 0:
                continue
    
            # estraiamo p ~ Dirichlet(alpha)
            proportions = np.random.dirichlet([alpha]*num_clients)
    
            # conteggi
            class_counts = (proportions * n_class).astype(int)
    
            gap = n_class - np.sum(class_counts)
            # distribuiamo l'eventuale gap
            while gap > 0:
                i = random.randint(0, num_clients-1)
                class_counts[i] += 1
                gap -= 1
    
            # Assegniamo i campioni di classe c
            start = 0
            for i in range(num_clients):
                count = class_counts[i]
                assigned_indices = idxs_c[start : start + count]
                client_indices[i].extend(assigned_indices)
                # Aggiorniamo la distribuzione
                agent_class_dist[i][c] += count
    
                start += count
        # Salva gli indici per ogni client
        for i in range(num_clients):
            torch.save(client_indices[i], client_indices_paths[i])
            print(f"[Info] Indici del client {i} salvati in {client_indices_paths[i]}.")
    # ------------------------------------------------
    # 5) Creiamo i DataLoader per i client
    # ------------------------------------------------
    client_loaders = []
    agent_classe = {}

    for i in range(num_clients):
        subset_i = Subset(dataset, client_indices[i])
        loader_i = DataLoader(subset_i, batch_size=batch_size, shuffle=shuffle)
        client_loaders.append(loader_i)
        class_counts = torch.zeros(num_classes, dtype=torch.long)

        total_per_client = len(subset_i)
        for _, labels in loader_i:
            # Conta le occorrenze di ciascuna classe nel batch
            batch_counts = torch.bincount(labels, minlength=num_classes)
            # Aggiungi i conteggi del batch al totale
            class_counts += batch_counts
        
        # Converti i conteggi in un dizionario
        distribution = {cls: int(count) for cls, count in enumerate(class_counts)}
        
        #approccio 1: ogni agente riceve sempre 10
        # agent_classe[i]=len(set(distribution.keys()))

        #approccio 2: ogni agente riceve il numero effettivo di classi, dove le istanze sono >0
        conta_classi = 0
        for key in distribution.keys():
            if distribution[key] >0:
                conta_classi+=1
        print(f"""[Client {i}] #campioni={total_per_client} -> Distribuzione classi:
        {distribution}""")
        agent_classe[i] = conta_classi
        

    server_subset = Subset(dataset, server_indices)
    server_loader = DataLoader(
        server_subset, batch_size=batch_size, shuffle=True
    )
    print(f"[Server] #campioni = {len(server_subset)} (fraction={server_fraction:.2f})")
    
    return client_loaders, server_loader, agent_classe


In [106]:
def save_checkpoint(state, filename="checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

In [107]:
def load_checkpoint(filename):
    print("=> Loading checkpoint")
    return  torch.load(filename)
   

In [108]:
test_case = 'base'
save_path = './dataloaders_indices'
num_clients = 25
server_data_fraction = 0.2 
batch_size = 16
alpha = 0.5
epochs = 5
num_rounds = 200
server_classes = 10
lambda_reg = 0.5
resume_from_checkpoint = False
check_point_name = './checkpoint/3_checkpoint.pth.tar'

In [109]:

transform = transforms.Compose([
     
    transforms.ToTensor(),
    transforms.Normalize(  mean=(0.49139968, 0.48215841, 0.44653091),
    std=(0.24703223, 0.24348513, 0.26158784))
])
dataset = CIFAR10(root='./datatest_training', train=True, download=True, transform=transform)
client_loaders, server_loader,agent_classes = make_federated_data(
        dataset=dataset,
        num_clients=num_clients,
        alpha=alpha,
        server_fraction=server_data_fraction,
        batch_size=batch_size,
        save_path = save_path
    )

test_dataset =   CIFAR10(root='./datatest_test', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)


Files already downloaded and verified


/tmp/ipykernel_224994/2406009991.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  server_indices = torch.load(server_indices_path)
/tmp/ipykernel_224994/2406009991.py:27

[Info] Indici caricati dai file di split.
[Client 0] #campioni=2227 -> Distribuzione classi:
        {0: 47, 1: 709, 2: 48, 3: 152, 4: 92, 5: 284, 6: 1, 7: 280, 8: 284, 9: 330}
[Client 1] #campioni=1475 -> Distribuzione classi:
        {0: 224, 1: 108, 2: 15, 3: 2, 4: 1, 5: 180, 6: 347, 7: 53, 8: 62, 9: 483}
[Client 2] #campioni=767 -> Distribuzione classi:
        {0: 9, 1: 2, 2: 24, 3: 37, 4: 13, 5: 494, 6: 67, 7: 46, 8: 5, 9: 70}
[Client 3] #campioni=1921 -> Distribuzione classi:
        {0: 2, 1: 1, 2: 634, 3: 1, 4: 312, 5: 67, 6: 8, 7: 496, 8: 0, 9: 400}
[Client 4] #campioni=913 -> Distribuzione classi:
        {0: 128, 1: 44, 2: 125, 3: 93, 4: 173, 5: 25, 6: 43, 7: 280, 8: 1, 9: 1}
[Client 5] #campioni=885 -> Distribuzione classi:
        {0: 1, 1: 313, 2: 272, 3: 160, 4: 65, 5: 0, 6: 39, 7: 4, 8: 5, 9: 26}
[Client 6] #campioni=1864 -> Distribuzione classi:
        {0: 371, 1: 23, 2: 465, 3: 194, 4: 8, 5: 323, 6: 71, 7: 99, 8: 245, 9: 65}
[Client 7] #campioni=1269 -> Distribuzion

In [ ]:
agents_confidence, aggregated_weights = federated_training_with_confidence(
    client_loaders=client_loaders,
    server_loader=server_loader,
    num_agents=num_clients,
    epochs=epochs,
    num_rounds=num_rounds,
    device='cuda',
    agent_classes = agent_classes,
    server_classes = server_classes,
    lambda_reg = lambda_reg,
    resume_from_checkpoint = resume_from_checkpoint,
    check_point_name = check_point_name
)

In [127]:
agent_weights, data_lengths, normalized_confidences, honest_clients = federated_training_with_confidence(
    client_loaders=client_loaders,
    server_loader=server_loader,
    num_agents=num_clients,
    epochs=epochs,
    num_rounds=num_rounds,
    device='cuda',
    agent_classes = agent_classes,
    server_classes = server_classes,
    lambda_reg = lambda_reg,
    resume_from_checkpoint = resume_from_checkpoint,
    check_point_name = check_point_name
)


Round 1/200 iniziato...
 Agente 0 inizia l'addestramento
 Agente 1 inizia l'addestramento
 Agente 2 inizia l'addestramento
 Agente 3 inizia l'addestramento
 Agente 4 inizia l'addestramento
 Agente 5 inizia l'addestramento
 Agente 6 inizia l'addestramento
 Agente 7 inizia l'addestramento
 Agente 8 inizia l'addestramento
 Agente 9 inizia l'addestramento
 Agente 10 inizia l'addestramento
 Agente 11 inizia l'addestramento
 Agente 12 inizia l'addestramento
 Agente 13 inizia l'addestramento
 Agente 14 inizia l'addestramento
 Agente 15 inizia l'addestramento
 Agente 16 inizia l'addestramento
 Agente 17 inizia l'addestramento
 Agente 18 inizia l'addestramento
 Agente 19 inizia l'addestramento
 Agente 20 inizia l'addestramento
 Agente 21 inizia l'addestramento
 Agente 22 inizia l'addestramento
 Agente 23 inizia l'addestramento
 Agente 24 inizia l'addestramento
agente 3 reweight regularization missing 1 -> 0.4
agente 5 reweight regularization missing 1 -> 0.4
agente 9 reweight regularization mi

In [111]:
            torch.cuda.empty_cache()


In [128]:

# 1. Calcolo dei pesi originali w_orig,i basati su data_lengths 
sum_data_lengths = sum(data_lengths[cid] for cid in honest_clients)
if sum_data_lengths == 0:
    raise ValueError("La somma delle lunghezze di tutti i client onesti è 0!")
print(sum_data_lengths)

19863


In [129]:

w_orig = {}
for cid in honest_clients:
    w_orig[cid] = data_lengths[cid] / sum_data_lengths
w_orig

{0: 0.11211800835724714,
 1: 0.07425867190253235,
 4: 0.045964859286109855,
 6: 0.09384282333987816,
 11: 0.0443538236922922,
 12: 0.07405729245330514,
 13: 0.1353773347429895,
 14: 0.13205457383074057,
 20: 0.10748628102502139,
 23: 0.044756582590746614,
 24: 0.1357297487791371}

In [130]:
# 2. Normalizzazione delle confidenze per ottenere r_norm,i 
confidences = normalized_confidences
sum_confidences = sum(confidences[cid] for cid in honest_clients)
print(sum_confidences)
if sum_confidences == 0:
    
    print("Attenzione: la somma delle confidenze è 0.")
    r_norm = {cid: 1.0 / len(honest_clients) for cid in honest_clients}
else:
    r_norm = {}
    for cid in honest_clients:
        r_norm[cid] = confidences[cid] / sum_confidences
r_norm

8.36585346445785


{0: 0.10211873118808391,
 1: 0.06813429155460342,
 4: 0.09622921699797438,
 6: 0.11953353046984133,
 11: 0.10101147782916484,
 12: 0.08565385707278127,
 13: 0.08579406811507032,
 14: 0.06517174862317915,
 20: 0.08232633722615731,
 23: 0.11271832492614078,
 24: 0.08130841599700343}

In [131]:

# --- 3. Calcolo dei pesi finali w_final,i ---
w_final = {}
for cid in honest_clients:
    w_final[cid] = r_norm[cid] * w_orig[cid]
w_final

{0: 0.011449348756777066,
 1: 0.005059562001864777,
 4: 0.004423162418524423,
 6: 0.011217363983073263,
 11: 0.0044802452785326595,
 12: 0.006343292742992561,
 13: 0.011614572278176716,
 14: 0.008606227490238075,
 20: 0.008848951818851425,
 23: 0.0050448870190474325,
 24: 0.011035970876902848}

In [132]:
agent_weights[0]['conv_block1.0.weight'][0]

KeyError: 'conv_block1.0.weight'

In [124]:
agent_weights[1]['conv_block1.0.weight'][0]

tensor([[[-0.0147, -0.1181,  0.1489],
         [-0.1242, -0.1567,  0.1088],
         [-0.1751, -0.0181,  0.1885]],

        [[ 0.0470,  0.0741, -0.1526],
         [-0.1373,  0.1427,  0.1478],
         [-0.1756, -0.0491, -0.1393]],

        [[-0.0842,  0.0494,  0.0625],
         [-0.0811, -0.0175, -0.1092],
         [-0.0610,  0.1933,  0.0306]]], device='cuda:0')

In [ ]:
agent_weights[0]['fc.weight'] [0]

In [ ]:
agent_weights[1]['fc.weight'] [0]

In [133]:
a = [x for x in agent_weights[0].keys() if 'classifier' in x]

In [66]:
a

['vgg.classifier.0.weight',
 'vgg.classifier.0.bias',
 'vgg.classifier.3.weight',
 'vgg.classifier.3.bias',
 'vgg.classifier.6.weight',
 'vgg.classifier.6.bias']

In [134]:
for x in a:
    print(agent_weights[0][x][0])

tensor([-0.0051, -0.0006, -0.0045,  ...,  0.0125,  0.0103,  0.0060],
       device='cuda:0')
tensor(0.0387, device='cuda:0')
tensor([-0.0180, -0.0116, -0.0049,  ...,  0.0035, -0.0049,  0.0025],
       device='cuda:0')
tensor(0.0794, device='cuda:0')
tensor([-0.0080,  0.0084, -0.0067,  ...,  0.0142, -0.0092,  0.0024],
       device='cuda:0')
tensor(0.0033, device='cuda:0')


In [68]:
for x in a:
    print(agent_weights[3][x][0])

tensor([-0.0051, -0.0006, -0.0045,  ...,  0.0125,  0.0103,  0.0060],
       device='cuda:0')
tensor(0.0387, device='cuda:0')
tensor([-0.0180, -0.0116, -0.0049,  ...,  0.0035, -0.0049,  0.0025],
       device='cuda:0')
tensor(0.0794, device='cuda:0')
tensor([-0.0043, -0.0108,  0.0042,  ..., -0.0040,  0.0088,  0.0122],
       device='cuda:0')
tensor(0.0031, device='cuda:0')


In [53]:
agent_weights[0]['vgg.features.0.weight'][0] * weight_factor

tensor([[[-13.9932, -19.7285, -12.1234],
         [  1.0173,  -1.2899,  -4.6679],
         [ 14.3288,  21.4251,  14.5492]],

        [[-21.3609, -30.9960, -20.7435],
         [  1.7713,   0.1598,  -4.5194],
         [ 21.1098,  32.5249,  21.5187]],

        [[ -9.0702, -14.4755,  -8.2167],
         [  0.3169,   0.7112,  -2.4775],
         [  8.2127,  16.1401,   9.3885]]], device='cuda:0')

In [59]:
data_lengths

{0: 2227,
 1: 1475,
 2: 767,
 3: 1921,
 4: 913,
 5: 885,
 6: 1864,
 7: 1269,
 8: 1784,
 9: 1805,
 10: 1714,
 11: 881,
 12: 1471,
 13: 2689,
 14: 2623,
 15: 999,
 16: 2841,
 17: 1205,
 18: 748,
 19: 1072,
 20: 2135,
 21: 2175,
 22: 952,
 23: 889,
 24: 2696}

In [50]:

# --- 4. Aggregazione dei pesi ---

numerator_sd = None
denominator = 0.0

for idx, cid in enumerate(honest_clients):
    local_sd = agent_weights[cid]  # state_dict del client i
    weight_factor = w_final[cid] * data_lengths[cid]
    denominator += weight_factor

    # Se è il primo client onesto, crea una copia profonda e scala
    if numerator_sd is None:
        numerator_sd = copy.deepcopy(local_sd)
        for key in numerator_sd:
            # print(key)
            numerator_sd[key] = (numerator_sd[key] * weight_factor)
    else:
        # Somma pesata in tutti i layer
        for key in local_sd:
            # print(key)

            numerator_sd[key] += local_sd[key] * weight_factor
    # print(numerator_sd['vgg.features.0.weight'][0])


In [51]:
for key in numerator_sd:
        numerator_sd[key] /= denominator

In [53]:
for x in a:
    print(agent_weights[0][x][0])

tensor([-0.0051, -0.0006, -0.0045,  ...,  0.0125,  0.0103,  0.0060],
       device='cuda:0')
tensor(0.0387, device='cuda:0')
tensor([-0.0180, -0.0116, -0.0049,  ...,  0.0035, -0.0049,  0.0025],
       device='cuda:0')
tensor(0.0794, device='cuda:0')
tensor([ 0.0132,  0.0077,  0.0034,  ..., -0.0129, -0.0021,  0.0049],
       device='cuda:0')
tensor(0.0051, device='cuda:0')


In [52]:
for x in a:
    print(numerator_sd[x][0])

tensor([-0.0051, -0.0006, -0.0045,  ...,  0.0125,  0.0103,  0.0060],
       device='cuda:0')
tensor(0.0387, device='cuda:0')
tensor([-0.0180, -0.0116, -0.0049,  ...,  0.0035, -0.0049,  0.0025],
       device='cuda:0')
tensor(0.0794, device='cuda:0')
tensor([-0.0066,  0.0043, -0.0008,  ..., -0.0027, -0.0027, -0.0010],
       device='cuda:0')
tensor(0.0030, device='cuda:0')


In [67]:
w_final

{0: 0.010370583408614489,
 1: 0.005335536760366712,
 4: 0.0040088972731093155,
 6: 0.008518808555584886,
 7: 0.0035892840596855537,
 11: 0.002966323700234828,
 12: 0.006318027737546384,
 13: 0.012062080110540314,
 18: 0.002348753466828463,
 20: 0.008315203658641392,
 22: 0.002828166004852138,
 23: 0.0035854262985643625,
 24: 0.00871600947426567}

In [68]:
sum(w_final.values())

0.0789631005088345

In [75]:
agent_weights[2]['vgg.features.0.weight'][0] * 1

tensor([[[-0.5955, -0.8396, -0.5159],
         [ 0.0433, -0.0549, -0.1986],
         [ 0.6098,  0.9118,  0.6192]],

        [[-0.9090, -1.3191, -0.8828],
         [ 0.0754,  0.0068, -0.1923],
         [ 0.8984,  1.3841,  0.9158]],

        [[-0.3860, -0.6160, -0.3497],
         [ 0.0135,  0.0303, -0.1054],
         [ 0.3495,  0.6869,  0.3995]]], device='cuda:0')

In [74]:
agent_weights[1]['vgg.features.0.weight'][0] * 1

tensor([[[-0.5955, -0.8396, -0.5159],
         [ 0.0433, -0.0549, -0.1986],
         [ 0.6098,  0.9118,  0.6192]],

        [[-0.9090, -1.3191, -0.8828],
         [ 0.0754,  0.0068, -0.1923],
         [ 0.8984,  1.3841,  0.9158]],

        [[-0.3860, -0.6160, -0.3497],
         [ 0.0135,  0.0303, -0.1054],
         [ 0.3495,  0.6869,  0.3995]]], device='cuda:0')

In [66]:
agent_weights[0]['vgg.features.0.weight'][0] * 1

tensor([[[-0.5955, -0.8396, -0.5159],
         [ 0.0433, -0.0549, -0.1986],
         [ 0.6098,  0.9118,  0.6192]],

        [[-0.9090, -1.3191, -0.8828],
         [ 0.0754,  0.0068, -0.1923],
         [ 0.8984,  1.3841,  0.9158]],

        [[-0.3860, -0.6160, -0.3497],
         [ 0.0135,  0.0303, -0.1054],
         [ 0.3495,  0.6869,  0.3995]]], device='cuda:0')

In [56]:
numerator_sd['vgg.features.0.weight'][0] / denominator

tensor([[[-0.5955, -0.8396, -0.5159],
         [ 0.0433, -0.0549, -0.1986],
         [ 0.6098,  0.9118,  0.6192]],

        [[-0.9090, -1.3191, -0.8828],
         [ 0.0754,  0.0068, -0.1923],
         [ 0.8984,  1.3841,  0.9158]],

        [[-0.3860, -0.6160, -0.3497],
         [ 0.0135,  0.0303, -0.1054],
         [ 0.3495,  0.6869,  0.3995]]], device='cuda:0')

In [ ]:
tensor([[[-0.5955, -0.8396, -0.5159],
         [ 0.0433, -0.0549, -0.1986],
         [ 0.6098,  0.9118,  0.6192]],

        [[-0.9090, -1.3191, -0.8828],
         [ 0.0754,  0.0068, -0.1923],
         [ 0.8984,  1.3841,  0.9158]],

        [[-0.3860, -0.6160, -0.3497],
         [ 0.0135,  0.0303, -0.1054],
         [ 0.3495,  0.6869,  0.3995]]], device='cuda:0')

In [ ]:




if denominator == 0:
    print("Attenzione: denominator = 0 durante l'aggregazione. Uso denominator=1 di default.")
    denominator = 1.0

# Dividiamo per la somma dei pesi
for key in numerator_sd:
    numerator_sd[key] /= denominator

 
# return numerator_sd


In [134]:
last_checkpoint_path

'../Carlo/vgg11_base_test/32x32 log c calcolato sulle classi dell agente/checkpoint/50_checkpoint.pth.tar'

In [ ]:
{0: 2, 1: 1, 2: 634, 3: 1, 4: 312, 5: 67, 6: 8, 7: 496, 8: 0, 9: 400}

In [90]:
client_loaders[3]

In [208]:
        
check_point = load_checkpoint('../Carlo/checkpoint/37_checkpoint.pth.tar')
device = 'cuda'
num_classes = 10
global_model = PretrainedVGG11(num_classes=10).to('cuda')

global_model.load_state_dict(check_point["state_dict"])
criterion = nn.CrossEntropyLoss(reduction = 'None')
global_model.eval()
global_model.to('cuda')
# log_C = np.log(num_classes)

total_sigma = 0.0
num_samples = 0
CM = np.zeros((10, 10), dtype=int)
class_count={}
with torch.no_grad():
    for data in client_loaders[3]:
        # print(f"num_samples {num_samples}")
        images, labels = data
        for i,label in enumerate(labels):
            # if label.cpu().item() in class_count.keys():
            #     class_count[label.cpu().item()]+=1
            # else:
            #     class_count[label.cpu().item()]=1

             if label.cpu().item() in class_count.keys():
                images[i] = images.to(device)
                labels[i] = labels.to(device)
                # print(losses)
                for loss_val in losses:
                    loss = loss_val.item()
                    # Calcola la confidenza
                    
                    max_term = max(-1 / np.exp(1), (loss - log_C) / lambda_reg)
                   
                        # print(f"loss - logC / lambda reg -> {(loss - log_C) / lambda_reg}")
                        # print(f" -2 / np.exp(1) -> {-2 / np.exp(1)}")
                        # print(f"max term {max_term}")
                       
                    W_input = 0.5 * max_term
                    W_output = lambertw(W_input).real
                    sigma_j = np.exp(-W_output)
                    total_sigma += sigma_j
                    num_samples += 1
                    # if max_print <=5:
                    #     max_print+=1
                    #     print(f"w input {W_input}")
                    #     print(f"w out {W_output}")
                    #     print(f"sigma_j {sigma_j}")









                
#         print(labels)
#         images = images.to(device)
#         labels = labels.to(device)
#         # Predizioni del modello
#         outputs = global_model(images)
#         probabilities = torch.softmax(outputs, dim=1)
#         # print(probabilities)
#         # losses = -torch.log(probabilities[range(len(labels)), labels])
#         # _, predicted = torch.max(outputs.data, 1)
#         _, preds = torch.max(outputs, 1)           
#         CM+=confusion_matrix(labels.cpu().numpy(), preds.cpu().numpy(),labels=range(num_classes))
# cm = CM
# from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
# y_true = []
# y_pred = []
# num_classes = cm.shape[0]
# for true_label in range(num_classes):
#     for pred_label in range(num_classes):
#         count = cm[true_label, pred_label]
#         y_true.extend([true_label] * count)
#         y_pred.extend([pred_label] * count)


# accuracy = accuracy_score(y_true, y_pred)
# precision, recall, f1, _ = precision_recall_fscore_support(
#     y_true, y_pred, labels=range(num_classes), average='weighted', zero_division=0
# )
# precision, recall, f1, accuracy

IndentationError: expected an indented block after 'with' statement on line 16 (980626496.py, line 17)

In [133]:
class_count

{7: 496, 4: 312, 9: 400, 2: 634, 5: 67, 6: 8, 0: 2, 1: 1, 3: 1}

In [221]:
check_point = load_checkpoint('../Carlo/checkpoint/37_checkpoint.pth.tar')


=> Loading checkpoint


/tmp/ipykernel_3894827/1589632707.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return  torch.load(filename)


In [365]:
        
device = 'cuda'
num_classes = 10
global_model = PretrainedVGG11(num_classes=10).to('cuda')

global_model.load_state_dict(check_point["state_dict"])
criterion = nn.CrossEntropyLoss(reduction = 'none')
global_model.eval()
global_model.to('cuda')
log_C = np.log(10)

total_sigma = 0.0
num_samples = 0
CM = np.zeros((10, 10), dtype=int)
class_count={}
yes = False
with torch.no_grad():
    for data in server_loader:
        # print(f"num_samples {num_samples}")
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        # print(labels)
        # for i,label in enumerate(labels):
            # if label.cpu().item() in class_count.keys():
            #     class_count[label.cpu().item()]+=1
            # else:
            #     class_count[label.cpu().item()]=1
    
             # if label.cpu().item() ==3:
             #    print(f"label: {label.cpu().item()} in position ",i)
        outputs = global_model(images)
        losses = criterion(outputs, labels)
        # for c,value in enumerate(losses):
        #     if c==i :print(  value.cpu().item())
        # print(labels)
        # print(losses)
        for loss_val in losses:
            loss = loss_val.item()
            # Calcola la confidenza
            
            max_term = max(-1 / np.exp(1), (loss - log_C) / lambda_reg)
           
                # print(f"loss - logC / lambda reg -> {(loss - log_C) / lambda_reg}")
                # print(f" -2 / np.exp(1) -> {-2 / np.exp(1)}")
                # print(f"max term {max_term}")
               
            W_input = 0.5 * max_term
            W_output = lambertw(W_input).real
            sigma_j = np.exp(-W_output)
            
            total_sigma += sigma_j
            num_samples += 1
            torch.cuda.empty_cache()

In [368]:
total_sigma/num_samples

1.0914749599020281

In [366]:
num_samples

10000

In [2]:
loss = 2.4325

In [346]:
 # #campioni=1921 -> Distribuzione classi:
 #        {0: 2, 1: 1, 2: 634, 3: 1, 4: 312, 5: 67, 6: 8, 7: 496, 8: 0, 9: 400}

In [347]:
-2 / np.exp(1) * 0.5

-0.36787944117144233

In [348]:
np.exp(-lambertw(-0.367879).real)

2.7140774650923585

In [7]:
(loss - np.log(9)) / 0.5

0.470550845327562

In [350]:
0.5 * max_term

-0.36787944117144233

In [11]:
log_C = np.log(9)
lambda_reg = 0.5
max_term = max(-2 / np.exp(1), (loss - log_C) / lambda_reg)
# print(f"loss - logC / lambda reg -> {(loss - log_C) / lambda_reg}")
# print(f" -2 / np.exp(1) -> {-2 / np.exp(1)}")
# print(f"max term {max_term}")

W_input = round(0.5 * max_term,10)
W_output = lambertw(W_input).real
sigma_j = np.exp(-W_output)
print(W_output)            


0.11571856813113592


In [241]:
 (loss - log_C) / 0.8

-1.7388723390952636

test client by client

In [427]:
local_model = PretrainedVGG11(num_classes=server_classes).to(device)
# local_model.load_state_dict(global_weights)

optimizer = optim.SGD(local_model.parameters(), lr=0.01,weight_decay = 1e-3)
criterion = nn.CrossEntropyLoss()
checkpoint = load_checkpoint('../Carlo/checkpoint/37_checkpoint.pth.tar')
local_model.load_state_dict(checkpoint['state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer'])
agente = 0
print(f"Resuming from checkpoint agent {agente}")
total_loss = 0
total_correct = 0
CM = np.zeros((10, 10), dtype=int)
epochs = 1
for epoch in range(epochs):
    print(f" Epoca {epoch} di {epochs}")

    local_model.train()
    for images, labels in client_loaders[agente]:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = local_model(images)
        loss = criterion(outputs, labels)
        

        loss.backward()
        torch.nn.utils.clip_grad_norm_(local_model.parameters(), 5)
        optimizer.step()
        total_loss += loss.item()
        predictions = outputs.argmax(dim=1)
        correct = (predictions == labels).sum().item()
        total_correct += correct
        CM+=confusion_matrix(labels.cpu().numpy(), predictions.cpu().numpy(),labels=range(num_classes))

        del images, labels, outputs
        torch.cuda.empty_cache()
 
cm = CM
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
y_true = []
y_pred = []
num_classes = cm.shape[0]
for true_label in range(num_classes):
    for pred_label in range(num_classes):
        count = cm[true_label, pred_label]
        y_true.extend([true_label] * count)
        y_pred.extend([pred_label] * count)


accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=range(num_classes), average='weighted', zero_division=0
)
print(precision, recall, f1, accuracy)

=> Loading checkpoint


/tmp/ipykernel_3894827/1589632707.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return  torch.load(filename)


Resuming from checkpoint agent 0
 Epoca 0 di 1
0.5696516082444957 0.5801526717557252 0.5719922431015887 0.5801526717557252


In [ ]:
agente 3:
5 epoche = 0.6432681727844674 0.6487246225923998 0.6418149074646156 0.6487246225923998
10 epoche = 0.64735889856957 0.6559604372722541 0.6482537836206366 0.6559604372722541
20 = 0.6628354806167412 0.6688443519000521 0.6625068788674227 0.6688443519000521
40 = 0.6775954213025263 0.6807261842790213 0.6763560593460642 0.6807261842790213


In [ ]:
agente 0:
5 = 0.5941672858450849 0.6041311180960934 0.5945540241220824 0.6041311180960934
10 epoche = 0.6067954257583104 0.6143691064211945 0.6060605355740651 0.6143691064211945
20 epoche = 0.6197980212277772 0.6264032330489447 0.6194148222440642 0.6264032330489447
40 epoche = 0.6357301581391942 0.6407611136057476 0.6353755168516289 0.6407611136057476
80 epoche = 0.6476813909045448 0.6509991019308486 0.6471684935335544 0.6509991019308486


In [430]:
        
device = 'cuda'
num_classes = 10
global_model = PretrainedVGG11(num_classes=10).to('cuda')
checkpoint = load_checkpoint('../Carlo/checkpoint/37_checkpoint.pth.tar')
global_model.load_state_dict(checkpoint['state_dict'])
# global_model.load_state_dict(local_model.state_dict())
criterion = nn.CrossEntropyLoss(reduction = 'none')
global_model.eval()
global_model.to('cuda')
log_C = np.log(10)
lambda_reg = 0.5
total_sigma = 0.0
num_samples = 0
CM = np.zeros((10, 10), dtype=int)
class_count={}
yes = False
with torch.no_grad():
    for data in server_loader:
        # print(f"num_samples {num_samples}")
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        # print(labels)
        # for i,label in enumerate(labels):
            # if label.cpu().item() in class_count.keys():
            #     class_count[label.cpu().item()]+=1
            # else:
            #     class_count[label.cpu().item()]=1
    
             # if label.cpu().item() ==3:
             #    print(f"label: {label.cpu().item()} in position ",i)
        outputs = global_model(images)
        losses = criterion(outputs, labels)
        predictions = outputs.argmax(dim=1)

        # for c,value in enumerate(losses):
        #     if c==i :print(  value.cpu().item())
        # print(labels)
        # print(losses)
        for loss_val in losses:
            loss = loss_val.item()
            # Calcola la confidenza
            
            max_term = max(-2 / np.exp(1), (loss - log_C) / lambda_reg)
           
                # print(f"loss - logC / lambda reg -> {(loss - log_C) / lambda_reg}")
                # print(f" -2 / np.exp(1) -> {-2 / np.exp(1)}")
                # print(f"max term {max_term}")
               
            W_input = round(0.5 * max_term,15)
            W_output = lambertw(W_input).real
            sigma_j = np.exp(-W_output)
            
            total_sigma += sigma_j
            num_samples += 1
            torch.cuda.empty_cache()
        CM+=confusion_matrix(labels.cpu().numpy(), predictions.cpu().numpy(),labels=range(num_classes))

        del images, labels, outputs
        torch.cuda.empty_cache()
         
cm = CM
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
y_true = []
y_pred = []
num_classes = cm.shape[0]
for true_label in range(num_classes):
    for pred_label in range(num_classes):
        count = cm[true_label, pred_label]
        y_true.extend([true_label] * count)
        y_pred.extend([pred_label] * count)


accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=range(num_classes), average='weighted', zero_division=0
)
print(precision, recall, f1, accuracy,total_sigma/num_samples)

=> Loading checkpoint


/tmp/ipykernel_3894827/1589632707.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return  torch.load(filename)


0.5632912919650324 0.5308 0.5266059616210143 0.5308 2.2242221620386817


In [ ]:
agente 3:
5 epoche = 0.405750174956724 0.2929 0.2029017867842363 0.2929 1.3088359905276843
10 epoche = 0.2947457935153108 0.3032 0.2046722368087383 0.3032 1.2909795089435945
20 epoche = 0.2572455295184062 0.3018 0.2020113360103667 0.3018 1.3036752163700225
40 epoche = 0.2654284063483688 0.2974 0.20148441614071072 0.2974 1.2704551316460677



In [ ]:
agente 0 = 
5 = 0.41264154730316593 0.4139 0.37316645754911026 0.4139 1.8698414305013973
80 epoche: 0.3920184284914333 0.3991 0.3486420046727647 0.3991 1.7128786229959845
10 epoche: 0.40849836538410683 0.4122 0.3621933447733203 0.4122 1.833366898753198
1 epoche : 0.5343381788412804 0.4193 0.3726221579632987 0.4193 1.9313442334733215
